In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import prince

from importlib.metadata import version
from pathlib import Path
from matplotlib.transforms import offset_copy

import io
import contextlib
from adjustText import adjust_text

In [2]:
%run LittRuP__import_functions.ipynb

In [3]:
# chemins vers fichiers Data et Images

BASE_DIR = Path.cwd()

if BASE_DIR.name == "Notebooks":
    BASE_DIR = BASE_DIR.parent

DAT_DIR = BASE_DIR / "Data"
IMG_DIR = BASE_DIR / "Images"
DOC_DIR = BASE_DIR / "Docs"

In [4]:
# import matrice réduite pour AC

matrix_reduced_profile = pd.read_csv(DAT_DIR / "LittRu_AC_matrix_reduced_profile.csv", sep=',', header=0)

In [5]:
# auteurs en index

matrix_reduced_profile = matrix_reduced_profile.set_index("Author")

In [6]:
matrix_reduced_profile.head()

,Amour,Apprentissage,Argent,Bonheur,Fantastique,Femme,Guerre,Intelligentsia,Liberté,Mort,Nature,Noblesse,Occident,Patrie,Révolution,Société,Souffrance,Traditions,Ville
Author,,,,,,,,,,,,,,,,,,,
A.OSTROVSKI,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
A.TOLSTOÏ,1,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0
AKSAKOV,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0
AVVAKUM,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
AÏTMATOV,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0


In [7]:
matrix_reduced_profile.shape

(40, 19)

**Refaire l'AC avec tous les axes possibles**

**Pour mesurer la qualité de représentation d’un point sur un axe ou un plan, on risque de ne pas évaluer correctement ce que représentent les axes 1–2 par rapport à l’ensemble de l’espace si on ne calcule que 2 axes.**

**AC complète**

In [8]:
# ============================================================
# 1. Préparation de la matrice pour l'AC complète
# ============================================================

X_CA = matrix_reduced_profile.copy()

# sécurité : suppression d'éventuelles lignes ou colonnes nulles
X_CA = X_CA.loc[
    X_CA.sum(axis=1) > 0,
    X_CA.sum(axis=0) > 0
]

# nombre maximal d'axes factoriels
n_axes = min(X_CA.shape) - 1

print("Nombre d'axes calculés :", n_axes)


# ============================================================
# 2. Analyse des correspondances complète
# ============================================================

ca = prince.CA(
    n_components=n_axes,
    n_iter=10,
    copy=True,
    check_input=True,
    engine="sklearn",
    random_state=0
)

ca = ca.fit(X_CA)


# ============================================================
# 3. Coordonnées principales
# ============================================================

# coordonnées des auteurs
row_coords = ca.row_coordinates(X_CA)

# coordonnées des thèmes
col_coords = ca.column_coordinates(X_CA)


# noms explicites des axes
axis_names = [
    f"Axe {axis_number}"
    for axis_number in range(1, row_coords.shape[1] + 1)
]

row_coords.columns = axis_names
col_coords.columns = axis_names


# ============================================================
# 4. Valeurs propres
# ============================================================

eig = pd.Series(
    ca.eigenvalues_,
    index=axis_names,
    name="Valeur propre"
)

# sécurité contre une éventuelle valeur propre nulle
eig_safe = eig.copy()
eig_safe[np.isclose(eig_safe, 0)] = np.nan


# ============================================================
# 5. Total général de la table
# ============================================================

grand_total = X_CA.to_numpy().sum()

print("Total général de la matrice :", grand_total)

Nombre d'axes calculés : 18
Total général de la matrice : 227


In [9]:
eig

Axe 1     0.545309
Axe 2     0.417646
Axe 3     0.398696
Axe 4     0.313371
Axe 5     0.270867
Axe 6     0.236455
Axe 7     0.216461
Axe 8     0.183778
Axe 9     0.153230
Axe 10    0.123948
Axe 11    0.118472
Axe 12    0.101477
Axe 13    0.069858
Axe 14    0.060880
Axe 15    0.043140
Axe 16    0.034728
Axe 17    0.023614
Axe 18    0.019164
Name: Valeur propre, dtype: float64

**Contributions des auteurs**

In [10]:
# ============================================================
# 6. Masses des auteurs
# ============================================================

row_masses = X_CA.sum(axis=1) / grand_total

row_masses.name = "Masse"
row_masses.index.name = "Author"

# vérification : la somme des masses doit être égale à 1
print(
    "Somme des masses des auteurs :",
    round(row_masses.sum(), 6)
)

# ============================================================
# 7. Contributions des auteurs sur tous les axes
# ============================================================

author_contributions_all = (
    row_coords
    .pow(2)
    .mul(row_masses, axis=0)
    .div(eig_safe, axis=1)
    .mul(100)
)

author_contributions_all.index.name = "Author"

# ============================================================
# 8. Contributions des auteurs sur le plan 1-2
# ============================================================

author_contributions_plan12 = author_contributions_all[
    ["Axe 1", "Axe 2"]
].copy()

author_contributions_plan12.columns = [
    "Contribution axe 1 (%)",
    "Contribution axe 2 (%)"
]

# inertie totale du plan 1-2
plan12_inertia = eig["Axe 1"] + eig["Axe 2"]


# contribution de chaque auteur à l'inertie du plan 1-2
author_contributions_plan12["Contribution plan 1-2 (%)"] = (
    row_masses
    * (
        row_coords["Axe 1"].pow(2)
        + row_coords["Axe 2"].pow(2)
    )
    / plan12_inertia
    * 100
)

# ajout de la masse de chaque auteur
author_contributions_plan12.insert(
    0,
    "Masse",
    row_masses
)

# classement décroissant selon la contribution au plan
author_contributions_plan12 = (
    author_contributions_plan12
    .sort_values(
        by="Contribution plan 1-2 (%)",
        ascending=False
    )
)

author_contributions_plan12.index.name = "Author"

# display(author_contributions_plan12.round(4))

# ============================================================
# 9. Export des contributions des auteurs
# ============================================================

# ------------------------------------------------------------
# 9.1. Export Excel : plusieurs feuilles
# ------------------------------------------------------------

output_file_authors = (
    DAT_DIR
    / "LittRu_AC_contributions_auteurs.xlsx"
)

with pd.ExcelWriter(
    output_file_authors,
    engine="openpyxl"
) as writer:

    author_contributions_all.to_excel(
        writer,
        sheet_name="Contributions tous axes"
    )

    author_contributions_plan12.to_excel(
        writer,
        sheet_name="Contributions plan 1-2"
    )

    row_masses.to_frame().to_excel(
        writer,
        sheet_name="Masses"
    )

    eig.to_frame().to_excel(
        writer,
        sheet_name="Valeurs propres"
    )

print(
    "Fichier Excel enregistré :",
    output_file_authors
)

# ------------------------------------------------------------
# 9.2. Exports CSV : un fichier par tableau
# ------------------------------------------------------------

output_csv_authors_all = (
    DAT_DIR
    / "LittRu_AC_contributions_auteurs_tous_axes.csv"
)

author_contributions_all.to_csv(
    output_csv_authors_all,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

output_csv_authors_plan12 = (
    DAT_DIR
    / "LittRu_AC_contributions_auteurs_plan_1-2.csv"
)

author_contributions_plan12.to_csv(
    output_csv_authors_plan12,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

output_csv_author_masses = (
    DAT_DIR
    / "LittRu_AC_masses_auteurs.csv"
)

row_masses.to_frame().to_csv(
    output_csv_author_masses,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

output_csv_eigenvalues = (
    DAT_DIR
    / "LittRu_AC_valeurs_propres.csv"
)

eig.to_frame().to_csv(
    output_csv_eigenvalues,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

print("Fichiers CSV enregistrés :")
print(output_csv_authors_all)
print(output_csv_authors_plan12)
print(output_csv_author_masses)
print(output_csv_eigenvalues)


Somme des masses des auteurs : 1.0
Fichier Excel enregistré : C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_contributions_auteurs.xlsx
Fichiers CSV enregistrés :
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_contributions_auteurs_tous_axes.csv
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_contributions_auteurs_plan_1-2.csv
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_masses_auteurs.csv
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_valeurs_propres.csv


In [10]:
# Contribution moyenne théorique d’un auteur

seuil_contrib_auteur = 100 / len(author_contributions_plan12)

print(
    "Contribution moyenne d'un auteur :",
    round(seuil_contrib_auteur, 4),
    "%"
)

Contribution moyenne d'un auteur : 2.5 %


**Contributions des thèmes**

In [11]:
# ============================================================
# 10. Masses des thèmes
# ============================================================

col_masses = X_CA.sum(axis=0) / grand_total

col_masses.name = "Masse"
col_masses.index.name = "Theme"

# vérification : la somme des masses doit être égale à 1
print(
    "Somme des masses des thèmes :",
    round(col_masses.sum(), 6)
)

# ============================================================
# 11. Contributions des thèmes sur tous les axes
# ============================================================

theme_contributions_all = (
    col_coords
    .pow(2)
    .mul(col_masses, axis=0)
    .div(eig_safe, axis=1)
    .mul(100)
)

theme_contributions_all.index.name = "Theme"

# ============================================================
# 12. Contributions des thèmes sur le plan 1-2
# ============================================================

theme_contributions_plan12 = theme_contributions_all[
    ["Axe 1", "Axe 2"]
].copy()

theme_contributions_plan12.columns = [
    "Contribution axe 1 (%)",
    "Contribution axe 2 (%)"
]

# contribution de chaque thème à l'inertie du plan 1-2
theme_contributions_plan12["Contribution plan 1-2 (%)"] = (
    col_masses
    * (
        col_coords["Axe 1"].pow(2)
        + col_coords["Axe 2"].pow(2)
    )
    / plan12_inertia
    * 100
)

# ajout de la masse de chaque thème
theme_contributions_plan12.insert(
    0,
    "Masse",
    col_masses
)

# classement décroissant selon la contribution au plan
theme_contributions_plan12 = (
    theme_contributions_plan12
    .sort_values(
        by="Contribution plan 1-2 (%)",
        ascending=False
    )
)

theme_contributions_plan12.index.name = "Theme"

# display(theme_contributions_plan12.round(4))

# ============================================================
# 13. Export des contributions des thèmes
# ============================================================

# ------------------------------------------------------------
# 13.1. Export Excel : plusieurs feuilles
# ------------------------------------------------------------

output_file_themes = (
    DAT_DIR
    / "LittRu_AC_contributions_themes.xlsx"
)

with pd.ExcelWriter(
    output_file_themes,
    engine="openpyxl"
) as writer:

    theme_contributions_all.to_excel(
        writer,
        sheet_name="Contributions tous axes"
    )

    theme_contributions_plan12.to_excel(
        writer,
        sheet_name="Contributions plan 1-2"
    )

    col_masses.to_frame().to_excel(
        writer,
        sheet_name="Masses"
    )

    eig.to_frame().to_excel(
        writer,
        sheet_name="Valeurs propres"
    )

print(
    "Fichier Excel enregistré :",
    output_file_themes
)

# ------------------------------------------------------------
# 13.2. Exports CSV : un fichier par tableau
# ------------------------------------------------------------

output_csv_themes_all = (
    DAT_DIR
    / "LittRu_AC_contributions_themes_tous_axes.csv"
)

theme_contributions_all.to_csv(
    output_csv_themes_all,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

output_csv_themes_plan12 = (
    DAT_DIR
    / "LittRu_AC_contributions_themes_plan_1-2.csv"
)

theme_contributions_plan12.to_csv(
    output_csv_themes_plan12,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

output_csv_theme_masses = (
    DAT_DIR
    / "LittRu_AC_masses_themes.csv"
)

col_masses.to_frame().to_csv(
    output_csv_theme_masses,
    sep=",",
    encoding="utf-8-sig",
    index=True
)

print("Fichiers CSV enregistrés :")
print(output_csv_themes_all)
print(output_csv_themes_plan12)
print(output_csv_theme_masses)

# il n'a pas été répété l’export CSV des valeurs propres, car eig est exactement le même tableau pour les auteurs et les thèmes. 
# Le fichier : LittRu_AC_valeurs_propres.csv
# créé dans la partie consacrée aux auteurs suffit pour toute l’AC.

Somme des masses des thèmes : 1.0
Fichier Excel enregistré : C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_contributions_themes.xlsx
Fichiers CSV enregistrés :
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_contributions_themes_tous_axes.csv
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_contributions_themes_plan_1-2.csv
C:\Users\thaly\OneDrive\Documents\_Post-Bac-Python\Programs (LittRu - Publication 2026.07.15)\Data\LittRu_AC_masses_themes.csv


NOTE: La colonne Contribution plan 1-2 (%) ne correspond pas à la simple addition des contributions aux axes 1 et 2. Elle mesure la contribution de chaque point à l’inertie totale du plan, en tenant compte des valeurs propres des deux axes.

In [12]:
# Contribution moyenne théorique d’un thème

seuil_contrib_theme = 100 / len(theme_contributions_plan12)

print(
    "Contribution moyenne d'un thème :",
    round(seuil_contrib_theme, 4),
    "%"
)

Contribution moyenne d'un thème : 5.2632 %


**Vérifications**

In [13]:
# Pour chaque axe, la somme des contributions des auteurs doit être égale à 100 %.

# vérification des sommes par axe
author_contribution_sums = author_contributions_all.sum(
    axis=0,
    min_count=1
)

display(author_contribution_sums.round(6))

Axe 1     100.0
Axe 2     100.0
Axe 3     100.0
Axe 4     100.0
Axe 5     100.0
Axe 6     100.0
Axe 7     100.0
Axe 8     100.0
Axe 9     100.0
Axe 10    100.0
Axe 11    100.0
Axe 12    100.0
Axe 13    100.0
Axe 14    100.0
Axe 15    100.0
Axe 16    100.0
Axe 17    100.0
Axe 18    100.0
dtype: float64

In [14]:
# Les trois colonnes doivent chacune totaliser environ 100 %.

author_plan12_sums = author_contributions_plan12[
    [
        "Contribution axe 1 (%)",
        "Contribution axe 2 (%)",
        "Contribution plan 1-2 (%)"
    ]
].sum()

display(author_plan12_sums.round(6))

Contribution axe 1 (%)       100.0
Contribution axe 2 (%)       100.0
Contribution plan 1-2 (%)    100.0
dtype: float64

In [15]:
# Pour chaque axe, la somme des contributions des thèmes doit être égale à 100 %.

# vérification des sommes par axe
theme_contribution_sums = theme_contributions_all.sum(
    axis=0,
    min_count=1
)

display(theme_contribution_sums.round(6))

Axe 1     100.0
Axe 2     100.0
Axe 3     100.0
Axe 4     100.0
Axe 5     100.0
Axe 6     100.0
Axe 7     100.0
Axe 8     100.0
Axe 9     100.0
Axe 10    100.0
Axe 11    100.0
Axe 12    100.0
Axe 13    100.0
Axe 14    100.0
Axe 15    100.0
Axe 16    100.0
Axe 17    100.0
Axe 18    100.0
dtype: float64

In [16]:
# Les trois colonnes doivent chacune totaliser environ 100 %.

theme_plan12_sums = theme_contributions_plan12[
    [
        "Contribution axe 1 (%)",
        "Contribution axe 2 (%)",
        "Contribution plan 1-2 (%)"
    ]
].sum()

display(theme_plan12_sums.round(6))

Contribution axe 1 (%)       100.0
Contribution axe 2 (%)       100.0
Contribution plan 1-2 (%)    100.0
dtype: float64